Aquí vamos a construir un modelo en estrella (Kimball) para almacenar los datos limpios. Originalmente pensé en montar todo en una tabla ancha en Dremio, pero para la Actividad se recomendaba un SQLite físico, así que armé este ETL paso a paso.

In [ ]:
import pandas as pd
import sqlite3
import os

Primero cargamos los datasets limpios del EDA. Tuve un error de rutas relativo al principio así que decidí dejar las rutas estáticas por seguridad.

In [ ]:
DIR_CLEAN = "data_clean"
terrazas_path = os.path.join(DIR_CLEAN, "Terrazas_202104_clean.csv")
licencias_path = os.path.join(DIR_CLEAN, "Licencias_Locales_202104_clean.csv")

df_terrazas = pd.read_csv(terrazas_path, sep=';', encoding='utf-8')
df_licencias = pd.read_csv(licencias_path, sep=';', encoding='utf-8')

Para las dimensiones, extraje valores únicos. Me topé con un problema raro: había terrazas sin una licencia registrada. Para no perder el registro en la tabla de hechos con un inner join, decidí crear un ID -1 de "Desconocido".

In [ ]:
# Dimensión Ubicación
ub_terrazas = df_terrazas[['desc_distrito_local', 'desc_barrio_local']].drop_duplicates().dropna().reset_index(drop=True)
dim_ubicacion = ub_terrazas.copy()
dim_ubicacion.insert(0, 'ID_Ubicacion', range(1, 1 + len(dim_ubicacion)))

# Dimensión Situación
sit_cols = ['desc_tipo_licencia', 'desc_tipo_situacion_licencia']
dim_situacion = df_licencias[sit_cols].drop_duplicates().dropna().reset_index(drop=True)
dim_situacion.insert(0, 'ID_Situacion', range(1, 1 + len(dim_situacion)))

# Manejo de registros sin licencia
unknown_sit = pd.DataFrame([[-1, 'Sin Licencia Registrada', 'N/A']], columns=['ID_Situacion', 'desc_tipo_licencia', 'desc_tipo_situacion_licencia'])
dim_situacion = pd.concat([unknown_sit, dim_situacion], ignore_index=True)

Armar la tabla de hechos fue un poco más complejo. Primero tuve que transformar las superficies de texto a números para poder sumarlas, y también me quedé solo con la licencia más reciente de cada local para no duplicar filas al cruzar.

In [ ]:
for col in ['Superficie_ES', 'Superficie_RA']:
    df_terrazas[col] = pd.to_numeric(df_terrazas[col], errors='coerce').fillna(0.0)

df_terrazas['Superficie_TO'] = df_terrazas['Superficie_ES'] + df_terrazas['Superficie_RA']

fact_df = df_terrazas.merge(dim_ubicacion, on=['desc_distrito_local', 'desc_barrio_local'], how='left')

# Nos quedamos con la última licencia registrada
licencias_unique = df_licencias.drop_duplicates(subset=['id_local'], keep='last')
fact_df = fact_df.merge(licencias_unique[['id_local'] + sit_cols], on='id_local', how='left')

fact_df = fact_df.merge(dim_situacion, on=sit_cols, how='left')
fact_df['ID_Situacion'] = fact_df['ID_Situacion'].fillna(-1).astype(int)

fact_cols = ['id_terraza', 'id_local', 'ID_Ubicacion', 'ID_Situacion', 'Superficie_ES', 'Superficie_RA', 'Superficie_TO']
fact_terrazas = fact_df[fact_cols]

Por último guardo esto en el archivo SQLite. Imprimo también las tablas guardadas para verificar que sí funcionó.

In [ ]:
db_path = "data_warehouse.db"
conn = sqlite3.connect(db_path)

dim_ubicacion.to_sql('Dim_Ubicacion', conn, if_exists='replace', index=False)
dim_situacion.to_sql('Dim_Situacion', conn, if_exists='replace', index=False)
fact_terrazas.to_sql('Fact_Terrazas', conn, if_exists='replace', index=False)

print(f"Dimensiones cargadas: Ubicacion ({len(dim_ubicacion)}) | Situacion ({len(dim_situacion)})")
print(f"Hechos: {len(fact_terrazas)} terrazas procesadas")

# Validamos la creación de tablas
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn))
conn.close()